# Tres en raya 3x4 (agente3_4)

Este notebook entrena un agente tabular para jugar a tres en raya en un tablero de 3 filas x 4 columnas (victoria: 3 en raya). El agente entrenado se guarda en `agente3_4.pickle`.

Estructura:
- Definición de `Board` para 3x4 con detección de 3 en raya.
- `Agent` y `Game` para self-play y entrenamiento.
- Entrenamiento y guardado de `agente3_4.pickle`.
- Uso/inferencia: jugar humano vs `agente3_4` cargado desde pickle.

In [2]:
import numpy as np
import pickle

In [3]:
class Board:
    def __init__(self, rows=3, cols=4, win_len=3):
        self.rows = rows
        self.cols = cols
        self.win_len = win_len
        self.state = np.zeros((rows, cols), dtype=int)

    def valid_moves(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols) if self.state[r, c] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] != 0:
            raise ValueError('movimiento ilegal')
        self.state[row, col] = symbol

    def reset(self):
        self.state = np.zeros((self.rows, self.cols), dtype=int)

    def is_game_over(self):
        # detecta 3 en raya en cualquier dirección (horizontal, vertical, diagonales)
        directions = [(0,1),(1,0),(1,1),(-1,1)]
        for r in range(self.rows):
            for c in range(self.cols):
                if self.state[r, c] == 0:
                    continue
                s = self.state[r, c]
                for dr, dc in directions:
                    coords = []
                    for k in range(self.win_len):
                        rr = r + dr*k
                        cc = c + dc*k
                        if 0 <= rr < self.rows and 0 <= cc < self.cols:
                            coords.append((rr, cc))
                        else:
                            break
                    if len(coords) == self.win_len:
                        total = sum(self.state[rr, cc] for rr, cc in coords)
                        if total == self.win_len * s:
                            return int(s)
        # empate
        if len(self.valid_moves()) == 0:
            return 0
        return None

In [4]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}
        self.alpha = alpha
        self.positions = []
        self.prob_exp = prob_exp
        self.symbol = None

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        if explore and np.random.uniform(0,1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]
        max_value = -1e9
        best = valid_moves[0]
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(board.rows * board.cols))
            value = self.value_function.get(next_state, 0)
            if value >= max_value:
                max_value = value
                best = (row, col)
        return best

    def update(self, board):
        self.positions.append(str(board.state.reshape(board.rows * board.cols)))

    def reward(self, reward):
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

In [5]:
class Game():
    def __init__(self, player1, player2):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Board()

    def selfplay(self, rounds=1000):
        wins = [0,0]
        for i in range(1, rounds+1):
            self.board.reset()
            for player in self.players:
                player.reset()
            game_over = False
            while not game_over:
                for player in self.players:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action[0], action[1])
                    for pl in self.players:
                        pl.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            winner = self.board.is_game_over()
            if winner == 0:
                for player in self.players:
                    player.reward(0.5)
            else:
                for idx, player in enumerate(self.players):
                    if winner == player.symbol:
                        player.reward(1)
                        wins[idx] += 1
                    else:
                        player.reward(0)
        return wins

In [10]:
# Entrenamiento rápido (ajusta rounds según tu tiempo)
agent1 = Agent(alpha=0.5, prob_exp=0.5)
agent2 = Agent()
game = Game(agent1, agent2)

wins = game.selfplay(10000)  # usa 10000 partidas por defecto
print('wins:', wins)

# inspeccionar top valores
items = sorted(agent1.value_function.items(), key=lambda kv: kv[1], reverse=True)
print('estados en la tabla:', len(items))
for s, v in items[:10]:
    print(v, s)

# guardar la funcion de valor
with open('agente3_4.pickle', 'wb') as f:
    pickle.dump(agent1.value_function, f, protocol=pickle.HIGHEST_PROTOCOL)

print('Guardado agente3_4.pickle')

wins: [6535, 3252]
estados en la tabla: 21786
1.0 [ 0 -1  0  0  0  0  0  0  1  1  1 -1]
1.0 [ 0  0  0 -1  0  0  0  0  1  1  1 -1]
1.0 [ 0 -1  0  0  0  0 -1  0  0  1  1  1]
1.0 [ 0  0  0  0  0  0 -1  0  1  1  1 -1]
0.999999999992724 [ 0  0  0  0  0 -1 -1  0  0  1  1  1]
0.9999999999854481 [ 0  0  0  0  0  0 -1  0 -1  1  1  1]
0.9999999999854481 [ 0  0  0  0  0  1  1  1  0 -1 -1  0]
0.9999999999854481 [ 0  0  0 -1  0  0  0 -1  1  1  1  0]
0.9999999999417923 [ 0  0  0  1  0  0  0  1  0 -1 -1  1]
0.9999999997671694 [ 0  0 -1  0  0  0 -1  0  0  1  1  1]
Guardado agente3_4.pickle


In [ ]:
# Inferencia: jugar contra el agente entrenado (robusto)
class AgentInferencia:
    def __init__(self, value_function, symbol=1):
        self.value_function = value_function
        self.symbol = symbol

    def move(self, board):
        valid_moves = board.valid_moves()
        max_value = -1e9
        best = valid_moves[0]
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(board.rows * board.cols))
            value = self.value_function.get(next_state, 0)
            if value >= max_value:
                max_value = value
                best = (row, col)
        return best


def dibujar_tablero(board):
    simbolos = {1: 'X', -1: 'O', 0: ' '}
    print('')
    for r in range(board.rows):
        fila = []
        for c in range(board.cols):
            if board.state[r, c] == 0:
                fila.append(str(r * board.cols + c + 1))
            else:
                fila.append(simbolos[int(board.state[r, c])])
        print(' ' + ' | '.join(fila))
        if r < board.rows - 1:
            print('---+' * (board.cols - 1) + '---')
    print('')


def movimiento_humano(board):
    valid_numbers = {r * board.cols + c + 1: (r, c) for r, c in board.valid_moves()}
    while True:
        entrada = input(f'Tu turno (elige {sorted(valid_numbers.keys())}): ').strip()
        if not entrada.isdigit():
            print('Entrada invalida. Escribe un numero.')
            continue
        numero = int(entrada)
        if numero not in valid_numbers:
            print('Esa casilla no esta disponible.')
            continue
        return valid_numbers[numero]


def jugar(agente_empieza=True, pickle_file='agente3_4.pickle', quick_train_episodes=1000):
    # intentamos cargar el pickle
    try:
        with open(pickle_file, 'rb') as f:
            func = pickle.load(f)
        print(f'Usando {pickle_file} cargado.')
    except Exception:
        # si no existe, comprobamos si hay un agente en memoria
        if 'agent1' in globals() and getattr(agent1, 'value_function', None) is not None:
            func = agent1.value_function
            print('Archivo pickle no encontrado: usando agent1 en memoria (value_function).')
        else:
            # entrenamos una version rapida para poder jugar
            print('No hay pickle ni agent1 en memoria. Entrenando una version rapida...')
            try:
                a1 = Agent(alpha=0.5, prob_exp=0.6)
                a2 = Agent()
                g = Game(a1, a2)
                wins = g.selfplay(quick_train_episodes)
                func = a1.value_function
                with open(pickle_file, 'wb') as f:
                    pickle.dump(func, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f'Entrenamiento rapido completado y guardado en {pickle_file}. wins={wins}')
            except Exception as e:
                print('Error entrenando rapidamente:', e)
                raise

    agente = AgentInferencia(func, symbol=1)
    simbolo_humano = -1
    board = Board()
    print('Humano: O | Agente: X')
    dibujar_tablero(board)
    turno_agente = agente_empieza
    while board.is_game_over() is None:
        if turno_agente:
            row, col = agente.move(board)
            board.update(agente.symbol, row, col)
            print(f'Agente juega en casilla {row * board.cols + col + 1}')
        else:
            row, col = movimiento_humano(board)
            board.update(simbolo_humano, row, col)
        dibujar_tablero(board)
        turno_agente = not turno_agente
    resultado = board.is_game_over()
    if resultado == 1:
        print('Gana el agente.')
    elif resultado == -1:
        print('Ganaste.')
    else:
        print('Empate.')

In [9]:
jugar(agente_empieza=False)

Humano: O | Agente: X

 1 | 2 | 3 | 4
---+---+---+---
 5 | 6 | 7 | 8
---+---+---+---
 9 | 10 | 11 | 12


 1 | O | 3 | 4
---+---+---+---
 5 | 6 | 7 | 8
---+---+---+---
 9 | 10 | 11 | 12

Agente juega en casilla 11

 1 | O | 3 | 4
---+---+---+---
 5 | 6 | 7 | 8
---+---+---+---
 9 | 10 | X | 12


 1 | O | 3 | 4
---+---+---+---
 5 | 6 | O | 8
---+---+---+---
 9 | 10 | X | 12

Agente juega en casilla 10

 1 | O | 3 | 4
---+---+---+---
 5 | 6 | O | 8
---+---+---+---
 9 | X | X | 12


 1 | O | 3 | 4
---+---+---+---
 5 | 6 | O | 8
---+---+---+---
 9 | X | X | O

Ganaste.


## Notas finales
- El agente es tabular: la llave de la `value_function` es la representación en string del `board.state.reshape(rows*cols)`.
- Ajusta `game.selfplay(...)` para más o menos entrenamiento.
- El archivo resultante `agente3_4.pickle` puede cargarse en otros entornos para jugar.